# 03 — Transformer Fine-Tuning (DeBERTa-v3)

**Goal:** Fine-tune DeBERTa-v3-base for fake news detection

**Target:** 95%+ accuracy, <100ms inference, zero external LLM dependency

**Hardware:** Kaggle P100 GPU (free tier) or Colab T4

**Runtime:** ~2-3 hours for 4 epochs

In [ ]:
# Install dependencies\n# !pip install transformers datasets accelerate evaluate onnx onnxruntime scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate
import torch
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load & Prepare Data

In [ ]:
# Load cleaned dataset
df = pd.read_csv('../backend/training/dataset_clean.csv')
print(f"Dataset size: {len(df)}")

# Split: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Convert to HuggingFace Dataset
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']]),
    'validation': Dataset.from_pandas(val_df[['text', 'label']]),
    'test': Dataset.from_pandas(test_df[['text', 'label']])
})

print(dataset)

## 2. Tokenization

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        padding='max_length'
    )

# Tokenize all splits
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print("Tokenization complete!")
print(tokenized_dataset)

## 3. Load Model

In [ ]:
# Binary classification: real (0) vs fake (1)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification"
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

## 4. Metrics

In [ ]:
# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    return {
        'accuracy': accuracy_metric.compute(predictions=predictions, references=labels)['accuracy'],
        'f1': f1_metric.compute(predictions=predictions, references=labels, average='macro')['f1'],
        'precision': precision_metric.compute(predictions=predictions, references=labels, average='macro')['precision'],
        'recall': recall_metric.compute(predictions=predictions, references=labels, average='macro')['recall']
    }

## 5. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./deberta-fakenews",
    
    # Training hyperparameters
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    # Optimization
    fp16=True,  # Mixed precision training (2x faster on GPU)
    gradient_accumulation_steps=2,
    
    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Logging
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",  # Disable wandb/tensorboard
    
    # Misc
    seed=42,
    push_to_hub=False
)

print("Training arguments configured!")

## 6. Train!

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting training...")
train_result = trainer.train()

print("\nTraining complete!")
print(f"Training time: {train_result.metrics['train_runtime']:.2f}s")
print(f"Samples/second: {train_result.metrics['train_samples_per_second']:.2f}")

## 7. Evaluation on Test Set

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(tokenized_dataset['test'])

print("\nTest Set Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Detailed predictions
predictions = trainer.predict(tokenized_dataset['test'])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'],
            yticklabels=['Real', 'Fake'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - DeBERTa-v3')
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))

## 8. Save Model

In [ ]:
# Save model and tokenizer
model.save_pretrained("./deberta-fakenews-final")
tokenizer.save_pretrained("./deberta-fakenews-final")

print("Model saved to: ./deberta-fakenews-final")

# Save metrics
import json
metrics = {
    'test_accuracy': test_results['eval_accuracy'],
    'test_f1': test_results['eval_f1'],
    'test_precision': test_results['eval_precision'],
    'test_recall': test_results['eval_recall'],
    'train_runtime': train_result.metrics['train_runtime'],
    'model_name': MODEL_NAME
}

with open('./deberta-fakenews-final/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nFinal metrics:")
print(json.dumps(metrics, indent=2))

## 9. ONNX Export (for Production)

In [ ]:
# Export to ONNX for fast CPU inference
from pathlib import Path
from transformers.onnx import export, FeaturesManager

# Get ONNX config
model_kind, model_onnx_config = FeaturesManager.check_supported_model_or_raise(model, feature='sequence-classification')
onnx_config = model_onnx_config(model.config)

# Export
onnx_path = Path("./deberta-fakenews-final/model.onnx")
export(
    preprocessor=tokenizer,
    model=model,
    config=onnx_config,
    opset=13,
    output=onnx_path
)

print(f"\nONNX model exported to: {onnx_path}")
print(f"File size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB")

## 10. Inference Speed Test

In [ ]:
import time

# Test inference speed
test_texts = test_df['text'].head(100).tolist()

start = time.time()
for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1)
end = time.time()

avg_time = (end - start) / len(test_texts) * 1000
print(f"\nAverage inference time: {avg_time:.1f}ms per sample")
print(f"Target: <100ms (achieved: {'✓' if avg_time < 100 else '✗'})")

## Summary

**DeBERTa-v3 Results:**
- Test Accuracy: [will be filled after training]
- Test F1: [will be filled after training]
- Inference Time: [will be filled after training]
- Model Size: ~350MB ONNX

**Next Steps:**
1. Upload model to HuggingFace Hub
2. Integrate ONNX model into backend (`backend/app/analysis/transformer.py`)
3. Replace external LLM calls in `api.py`
4. Benchmark end-to-end latency